In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from google.colab import files
from google.colab import drive; drive.mount('/content/drive')
import calendar
import os

# ---------------------------------------------------------
# CONFIG - Ruta del archivo Excel
# ---------------------------------------------------------
archivo= "/content/drive/MyDrive/Sales BILANNATURALS 2023-Mar2026.xlsx"
NOMBRE_HOJA = "Reporte Ventas"

# ---------------------------------------------------------
# FUNCIONES
# ---------------------------------------------------------

def es_ultimo_dia_habil(fecha):
    """
    Retorna True si la fecha es el último día hábil del mes.
    Incluye sábado como día hábil.
    """
    año = fecha.year
    mes = fecha.month
    ultimo_dia_mes = calendar.monthrange(año, mes)[1]
    fecha_ultimo = datetime(año, mes, ultimo_dia_mes)

    # Si el último día cae domingo, retrocede a sábado
    if fecha_ultimo.weekday() == 6:  # domingo
        fecha_ultimo -= timedelta(days=1)

    return fecha.date() == fecha_ultimo.date()


# ---------------------------------------------------------
# CARGA DEL EXCEL
# ---------------------------------------------------------
df = pd.read_excel(archivo, sheet_name=NOMBRE_HOJA)

# Normalizar columnas
df.columns = df.columns.str.strip().str.lower()

# Convertir la columna de fecha
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors="coerce")

# Eliminar filas inválidas
df = df.dropna(subset=["fecha", "cliente"])

# ---------------------------------------------------------
# OBTENER ÚLTIMA COMPRA DEL CLIENTE
# ---------------------------------------------------------
ultima_compra = (
    df.groupby("cliente")["fecha"]
    .max()
    .reset_index()
    .rename(columns={"fecha": "ultima_fecha_compra"})
)

# Asegurar que la columna sea datetime
ultima_compra["ultima_fecha_compra"] = pd.to_datetime(ultima_compra["ultima_fecha_compra"])

# ---------------------------------------------------------
# CALCULAR DELTA DE DÍAS
# ---------------------------------------------------------
fecha_actual = pd.to_datetime(datetime.now())

ultima_compra["delta_dias"] = (fecha_actual - ultima_compra["ultima_fecha_compra"]).dt.days

# ---------------------------------------------------------
# FILTRAR CLIENTES 25 – 366 DÍAS SIN COMPRAR
# ---------------------------------------------------------
clientes_recompra = ultima_compra[
    (ultima_compra["delta_dias"] >= 25) &
    (ultima_compra["delta_dias"] <= 366)]

# Si no hay clientes, no genera archivo
if clientes_recompra.empty:
    print("No hay clientes con delta entre 25 y 365 días.")
    exit()

# ---------------------------------------------------------
# GENERAR NOMBRE DEL ARCHIVO
# ---------------------------------------------------------
fecha_inicio = (fecha_actual - timedelta(days=365)).strftime("%d %b %Y").lower()
fecha_fin = (fecha_actual - timedelta(days=25)).strftime("%d %b %Y").lower()

nombre_archivo = f"Clientes 25-365 dias sin comprar {fecha_inicio} - {fecha_fin}.xlsx"
nombre_archivo = nombre_archivo.replace("á","a").replace("é","e").replace("í","i").replace("ó","o").replace("ú","u")

# ---------------------------------------------------------
# EXPORTAR RESULTADO
# ---------------------------------------------------------
clientes_recompra.to_excel(nombre_archivo, index=False)

print(f"Archivo generado correctamente: {nombre_archivo}")

# ---------------------------------------------------------
# OPCIONAL: Validar si hoy es último día hábil del mes
# ---------------------------------------------------------
if es_ultimo_dia_habil(fecha_actual):
    print("Hoy es el último día hábil del mes. Proceso ejecutado.")
else:
    print("Proceso ejecutado en modo diario.")

files.download(nombre_archivo)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivo generado correctamente: Clientes 25-365 dias sin comprar 11 dec 2024 - 16 nov 2025.xlsx
Proceso ejecutado en modo diario.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>